In [1]:
import ast
import numpy as np
import pandas as pd
import tqdm as notebook_tqdm
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel

from sklearn.model_selection import train_test_split
from tqdm import tqdm

## Label maps

In [2]:
NER_TAGS = [
    'B-ACCOUNTNAME', 'B-ACCOUNTNUMBER', 'B-CREDITCARDNUMBER', 'B-EMAIL',
    'B-IP', 'B-IPV4', 'B-IPV6', 'B-MAC', 'B-PASSWORD', 'B-PHONE_NUMBER',
    'B-SSN', 'B-USERNAME',
    'I-ACCOUNTNAME', 'I-ACCOUNTNUMBER', 'I-CREDITCARDNUMBER', 'I-EMAIL',
    'I-IP', 'I-IPV4', 'I-IPV6', 'I-MAC', 'I-PASSWORD', 'I-PHONE_NUMBER',
    'I-SSN', 'I-USERNAME', 'O'
]
tag2idx = {tag: i for i, tag in enumerate(NER_TAGS)}
idx2tag = {i: tag for tag, i in tag2idx.items()}
NUM_TAGS = len(NER_TAGS)

## Tokenizer

The dataset was pre-tokenized with `distilbert-base-uncased`, so we load
the same tokenizer here — only to call `convert_tokens_to_ids` and to
obtain the correct `pad_token_id` (0).  We do **not** re-tokenize.

In [3]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

## Load data

In [4]:
df = pd.read_parquet('/kaggle/input/datasets/abdoamin12/pii-data/data.parquet')
df = pd.DataFrame(df)
df.head()

,masked_text,unmasked_text,token_entity_labels,tokenised_unmasked_text
0,[PREFIX_1] [FIRSTNAME_1] [MIDDLENAME_1] [LASTN...,"Mr. Adolphus Reagan Ziemann, as a Central Prin...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...","[mr, ., adolph, ##us, reagan, z, ##ie, ##mann,..."
1,"Hello [FIRSTNAME_1], would you please investig...","Hello Hannah, would you please investigate the...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...","[hello, hannah, ,, would, you, please, investi..."
2,We also request a review of our policies with ...,We also request a review of our policies with ...,"[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...","[we, also, request, a, review, of, our, polici..."
3,"Dear [FIRSTNAME_1], a company-wide presentatio...","Dear Devan, a company-wide presentation is req...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...","[dear, dev, ##an, ,, a, company, -, wide, pres..."
4,Can we also have a session on how to manage st...,Can we also have a session on how to manage st...,"[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...","[can, we, also, have, a, session, on, how, to,..."


In [5]:
# Drop rows where token list and label list have different lengths
before = len(df)
df = df[
    df.apply(
        lambda r: len(r["tokenised_unmasked_text"]) == len(r["token_entity_labels"]),
        axis=1
    )
].reset_index(drop=True)
after = len(df)
print(f"Removed {before - after} rows")

Removed 190 rows


In [6]:
train_df, test_df = train_test_split(df, test_size=0.1, random_state=42)
train_df, val_df  = train_test_split(train_df, test_size=0.1, random_state=42)
print(len(train_df), len(val_df), len(test_df))

17332 1926 2140


In [7]:
tokens = train_df.iloc[0]["tokenised_unmasked_text"]
ids    = tokenizer.convert_tokens_to_ids(tokens)
print(tokens[:10])
print(ids[:10])

['in' 'your' 'last' 'therapy' 'session' ',' 'you' 'had' 'some' 'questions']
[1999, 2115, 2197, 7242, 5219, 1010, 2017, 2018, 2070, 3980]


In [8]:
all_tags = set()
for labels in df["token_entity_labels"]:
    all_tags.update(labels)

for tag in NER_TAGS:
    if tag not in all_tags:
        print(f"Tag {tag} not found in dataset!")

In [9]:
from collections import Counter

counter = Counter()
for labels in train_df["token_entity_labels"]:
    counter.update(labels)
print(counter.most_common())

[('O', 1028781), ('I-IPV6', 23752), ('I-IP', 13801), ('I-EMAIL', 9731), ('I-MAC', 9219), ('I-PHONE_NUMBER', 8326), ('I-CREDITCARDNUMBER', 6762), ('I-PASSWORD', 6377), ('I-IPV4', 5688), ('I-USERNAME', 4036), ('I-SSN', 3631), ('I-ACCOUNTNUMBER', 3259), ('I-ACCOUNTNAME', 1404), ('B-EMAIL', 1162), ('B-USERNAME', 1086), ('B-IPV4', 948), ('B-ACCOUNTNUMBER', 888), ('B-PASSWORD', 846), ('B-ACCOUNTNAME', 844), ('B-PHONE_NUMBER', 839), ('B-IPV6', 810), ('B-CREDITCARDNUMBER', 791), ('B-IP', 770), ('B-MAC', 733), ('B-SSN', 633)]


## Model


In [10]:
import torch
import torch.nn as nn
from transformers import DistilBertModel

class DistilBERTBiLSTM(nn.Module):

    def __init__(self, num_tags, ignore_index=-100):
        super().__init__()

        self.bert = DistilBertModel.from_pretrained("distilbert-base-uncased")

        self.lstm = nn.LSTM(
            input_size=768,
            hidden_size=256,
            batch_first=True,
            bidirectional=True
        )
        self.classifier = nn.Linear(512, num_tags)
        self.loss_fn = nn.CrossEntropyLoss(ignore_index=ignore_index)

    def forward(self, input_ids, attention_mask, labels=None):
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask.float()
        )

        hidden   = outputs.last_hidden_state   # (B, T, 768)
        lstm_out, _ = self.lstm(hidden)        # (B, T, 512)
        logits   = self.classifier(lstm_out)   # (B, T, num_tags)

        if labels is None:
            return logits

        loss = self.loss_fn(
            logits.view(-1, logits.shape[-1]),
            labels.view(-1)
        )
        return loss, logits

## Dataset and DataLoader


In [11]:
from torch.utils.data import Dataset

class PIIBertDataset(Dataset):

    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        tokens    = row["tokenised_unmasked_text"]
        labels    = row["token_entity_labels"]
        input_ids = tokenizer.convert_tokens_to_ids(tokens)
        label_ids = [tag2idx[t] for t in labels]

        return input_ids, label_ids

In [12]:
import torch
from torch.nn.utils.rnn import pad_sequence

IGNORE_INDEX = -100

def collate_fn(batch):
    token_ids = [torch.tensor(x[0], dtype=torch.long) for x in batch]
    label_ids = [torch.tensor(x[1], dtype=torch.long) for x in batch]

    token_ids = pad_sequence(token_ids, batch_first=True, padding_value=tokenizer.pad_token_id)
    label_ids = pad_sequence(label_ids, batch_first=True, padding_value=IGNORE_INDEX)

    mask = (token_ids != tokenizer.pad_token_id)

    return token_ids, label_ids, mask

In [13]:
from torch.utils.data import DataLoader

train_dataset = PIIBertDataset(train_df)
val_dataset   = PIIBertDataset(val_df)
test_dataset  = PIIBertDataset(test_df)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,  collate_fn=collate_fn)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False, collate_fn=collate_fn)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False, collate_fn=collate_fn)

In [14]:
# Shape sanity-check
tokens, labels, mask = next(iter(train_loader))
print("tokens:", tokens.shape)
print("labels:", labels.shape)
print("mask  :", mask.shape)

tokens: torch.Size([32, 128])
labels: torch.Size([32, 128])
mask  : torch.Size([32, 128])


## Training utilities

In [15]:
def overfit_one_batch(model, dataset, batch_size=16, epochs=200, lr=1e-4):

    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
    x, y, mask = next(iter(loader))

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    x, y, mask = x.to(device), y.to(device), mask.to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()

        loss, logits = model(input_ids=x, attention_mask=mask, labels=y)
        loss.backward()
        optimizer.step()

        if epoch % 10 == 0:
            preds = logits.argmax(-1)
            valid = mask & (y != IGNORE_INDEX)
            acc   = (preds[valid] == y[valid]).float().mean()
            print(f"Epoch {epoch:3d} | Loss {loss.item():.4f} | Acc {acc:.4f}")

In [16]:
from tqdm import tqdm

def evaluate(model, loader, device):
    model.eval()
    total_loss = 0
    correct    = 0
    total      = 0

    with torch.no_grad():
        for x, y, mask in tqdm(loader, desc="Evaluating"):
            x, y, mask = x.to(device), y.to(device), mask.to(device)

            loss, logits = model(input_ids=x, attention_mask=mask, labels=y)
            total_loss  += loss.item()

            preds = logits.argmax(dim=-1)
            valid    = mask & (y != IGNORE_INDEX)
            correct += (preds[valid] == y[valid]).sum().item()
            total   += valid.sum().item()

    avg_loss= total_loss / len(loader)
    acc= correct / total if total > 0 else 0.0
    return avg_loss, acc

In [17]:
def train(
    model,
    train_dataset,
    val_dataset,
    batch_size=16,
    epochs=5,
    lr=2e-5,
    patience=2
):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,  collate_fn=collate_fn)
    val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

   
    optimizer = torch.optim.AdamW([
        {"params": model.bert.parameters(),       "lr": lr},
        {"params": model.lstm.parameters(),       "lr": lr * 10},
        {"params": model.classifier.parameters(), "lr": lr * 10},
    ])

    best_loss  = float("inf")
    no_improve = 0

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0

        for x, y, mask in tqdm(train_loader, desc=f"Epoch {epoch}"):
            x, y, mask = x.to(device), y.to(device), mask.to(device)

            optimizer.zero_grad()
            loss, _ = model(input_ids=x, attention_mask=mask, labels=y)
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item()

        train_loss          = total_loss / len(train_loader)
        val_loss, val_acc   = evaluate(model, val_loader, device)

        print(
            f"Epoch {epoch:2d} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val Acc: {val_acc:.4f}"
        )

        if val_loss < best_loss:
            best_loss  = val_loss
            no_improve = 0
            torch.save(model.state_dict(), "best.pt")
        else:
            no_improve += 1
            if no_improve >= patience:
                print("Early stopping")
                break

    model.load_state_dict(torch.load("best.pt"))
    return model

## Instantiate model

In [22]:
model = DistilBERTBiLSTM(num_tags=NUM_TAGS)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [23]:
# FIX: move model to device BEFORE running the forward pass
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

with torch.no_grad():
    x, y, mask = next(iter(val_loader))
    x    = x.to(device)
    mask = mask.to(device)

    logits = model(input_ids=x, attention_mask=mask)   # no labels → returns logits only
    preds  = logits.argmax(-1)

print("Unique predicted tag indices:", torch.unique(preds))

Unique predicted tag indices: tensor([ 0,  1,  2,  4,  5,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 20,
        21, 22, 23, 24], device='cuda:0')


## Optional: overfit sanity-check

Uncomment to verify the model can memorise one batch before full training.

In [24]:
# dataset = PIIBertDataset(train_df)
# overfit_one_batch(model, dataset, batch_size=16, epochs=300, lr=1e-4)

## Train

`lr=2e-5` is the standard fine-tuning rate for DistilBERT.  The BiLSTM
and classifier heads receive `lr * 10 = 2e-4` via the param groups above.

In [25]:
trained_model = train(
    model,
    train_dataset,
    val_dataset,
    batch_size=32,
    epochs=20,
    lr=2e-5,       
    patience=3
)

Evaluating: 100%|██████████| 61/61 [00:09<00:00,  6.44it/s]


Epoch  1 | Train Loss: 0.1413 | Val Loss: 0.0371 | Val Acc: 0.9808


Evaluating: 100%|██████████| 61/61 [00:09<00:00,  6.47it/s]


Epoch  2 | Train Loss: 0.0343 | Val Loss: 0.0297 | Val Acc: 0.9859


Evaluating: 100%|██████████| 61/61 [00:09<00:00,  6.46it/s]


...

Epoch  4 | Train Loss: 0.0213 | Val Loss: 0.0283 | Val Acc: 0.9850


Evaluating: 100%|██████████| 61/61 [00:09<00:00,  6.47it/s]


Epoch  5 | Train Loss: 0.0194 | Val Loss: 0.0280 | Val Acc: 0.9869


Evaluating: 100%|██████████| 61/61 [00:09<00:00,  6.47it/s]


Epoch  6 | Train Loss: 0.0147 | Val Loss: 0.0349 | Val Acc: 0.9870
Early stopping


## Evaluation

In [26]:
from tqdm import tqdm
from sklearn.metrics import classification_report

def test_evaluate(model, test_loader, device):

    model.eval()

    sent_preds  = []
    sent_labels = []

    with torch.no_grad():
        for x, y, mask in tqdm(test_loader, desc="Testing"):
            x    = x.to(device)
            y    = y.to(device)
            mask = mask.to(device)

            logits = model(input_ids=x, attention_mask=mask)   
            preds  = logits.argmax(dim=-1)

            for pred_seq, true_seq, m_seq in zip(preds, y, mask):
                valid     = m_seq.bool()
                pred_list = pred_seq[valid].cpu().tolist()
                true_list = true_seq[valid].cpu().tolist()
                sent_preds.append(pred_list)
                sent_labels.append(true_list)

    return sent_labels, sent_preds

In [27]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
sent_labels, sent_preds = test_evaluate(trained_model, test_loader, device)

# Flatten for sklearn token-level report
flat_labels = [t for seq in sent_labels for t in seq]
flat_preds  = [t for seq in sent_preds  for t in seq]

print(classification_report(
    flat_labels,
    flat_preds,
    labels=list(range(NUM_TAGS)),
    target_names=[idx2tag[i] for i in range(NUM_TAGS)],
    zero_division=0
))

Testing: 100%|██████████| 67/67 [00:09<00:00,  6.72it/s]


                    precision    recall  f1-score   support

     B-ACCOUNTNAME       0.99      0.95      0.97        95
   B-ACCOUNTNUMBER       0.97      0.98      0.98       118
B-CREDITCARDNUMBER       0.71      0.92      0.80        89
           B-EMAIL       1.00      1.00      1.00       157
              B-IP       0.46      0.44      0.45        94
            B-IPV4       0.73      0.73      0.73       127
            B-IPV6       0.81      0.84      0.82        91
             B-MAC       0.98      0.97      0.97        94
        B-PASSWORD       0.99      0.97      0.98       113
    B-PHONE_NUMBER       0.99      0.98      0.99       117
             B-SSN       1.00      0.99      0.99        82
        B-USERNAME       0.94      0.96      0.95       135
     I-ACCOUNTNAME       0.99      0.95      0.97       156
   I-ACCOUNTNUMBER       0.97      1.00      0.98       422
I-CREDITCARDNUMBER       0.71      0.94      0.81       761
           I-EMAIL       1.00      1.00

In [28]:
!pip install seqeval -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [29]:
from seqeval.metrics import classification_report as seqeval_report
from seqeval.metrics import f1_score


true_tags = [[idx2tag[t] for t in seq] for seq in sent_labels]
pred_tags = [[idx2tag[t] for t in seq] for seq in sent_preds]

print(seqeval_report(true_tags, pred_tags))
print("Span-level F1:", f1_score(true_tags, pred_tags))

                  precision    recall  f1-score   support

     ACCOUNTNAME       0.96      0.93      0.94        95
   ACCOUNTNUMBER       0.95      0.99      0.97       118
CREDITCARDNUMBER       0.68      0.92      0.78        89
           EMAIL       0.99      0.99      0.99       157
              IP       0.19      0.36      0.25        94
            IPV4       0.56      0.61      0.59       127
            IPV6       0.39      0.58      0.47        91
             MAC       0.97      1.00      0.98        94
        PASSWORD       0.96      0.96      0.96       113
    PHONE_NUMBER       0.99      0.99      0.99       117
             SSN       0.99      0.99      0.99        82
        USERNAME       0.88      0.96      0.92       135

       micro avg       0.76      0.87      0.81      1312
       macro avg       0.79      0.86      0.82      1312
    weighted avg       0.81      0.87      0.83      1312

Span-level F1: 0.8092526690391458
